In [4]:
import pandas as pd
import numpy as np
import os
import json
import glob
from itertools import combinations
from collections import defaultdict

from matplotlib import pyplot as plt

In [ ]:
# Main data file
df = pd.read_csv('../data/evaluations.csv', sep='\t')
df.head()

### Create prompts that are 1 reviewer vs all the papers they have reviewed (10)

In [ ]:
cur_dir = os.getcwd()
dic = {}
data_dir = '../evaluation_datasets/'

for filename in os.scandir(data_dir):
    reviewer_folder = filename.path + '/archives/'
    paper_file = filename.path + '/submissions.json'

    if not os.path.exists(paper_file):
        continue
    # all paper content
    paper_content = json.load(open(paper_file))
    # all possible papers
    papers = list(paper_content.keys())
    
    os.makedirs(f'{filename.path}/prompts', exist_ok=True)
    os.makedirs(f'{filename.path}/prompts_gt', exist_ok=True)
    for file in os.scandir(reviewer_folder):
        reviewer_published_papers, reviewer_expertise_papers = None, None
        reviewer = int(file.name.strip('.jsonl')[1:])
        with open(file) as f:
            reviewer_published_papers = [json.loads(line) for line in f]
            
            row = df[df['ParticipantID'] == reviewer].iloc[0]
            reviewer_expertise_papers =  {row[f'Paper{x}']: row[f'Expertise{x}'] for x in range(1, 11)
                                            if not pd.isna(row[f'Paper{x}'])}
        
        with open(f'{filename.path}/prompts/{reviewer}.txt', 'w') as f:
            f.write('[Reviewer papers:\n')
            for i, p in enumerate(reviewer_published_papers):
                f.write(f'{i+1}. Title: {p["content"]["title"]}\n')
                f.write(f'Abstract: {p["content"]["abstract"]}\n')
            f.write(']\n')
            f.write('=====================\n')
            f.write('New papers:\n')
            for i, p in enumerate(reviewer_expertise_papers.keys()):
                expertise_paper_info = paper_content.get(p, None)
                if expertise_paper_info is None:
                    continue
                f.write(f'{i+1}. <{p}: Title: {expertise_paper_info["content"]["title"]}\n')
                f.write(f'Abstract: {expertise_paper_info["content"]["abstract"]} >\n')
        
        with open(f'{filename.path}/prompts_gt/{reviewer}.txt', 'w') as f:
            f.write('[Reviewer papers:\n')
            for i, p in enumerate(reviewer_published_papers):
                f.write(f'{i+1}. Title: {p["content"]["title"]}\n')
                f.write(f'Abstract: {p["content"]["abstract"]}\n')
            f.write(']\n')
            f.write('=====================\n')
            f.write('New papers:\n')
            for i, p in enumerate(reviewer_expertise_papers.keys()):
                expertise_paper_info = paper_content.get(p, None)
                if expertise_paper_info is None:
                    continue
                f.write(f'{i+1}. <{p}: Title: {expertise_paper_info["content"]["title"]}\n')
                f.write(f'Abstract: {expertise_paper_info["content"]["abstract"]} >\n')
            f.write('=====================\n')
            f.write('Output Scores:\n')
            for i, p in enumerate(reviewer_expertise_papers.keys()):
                f.write(f'{i+1}.{p}: {reviewer_expertise_papers[p]}\n')
        
    break    
    

### Get the paper and reviewer data in the desired format

In [79]:
# get the reviewer and paper information for each dataset
cur_dir = os.getcwd()
dic = {}
data_dir = '../evaluation_datasets/'

for filename in os.scandir(data_dir):
    reviewer_folder = filename.path + '/archives/'
    paper_file = filename.path + '/submissions.json'

    if not os.path.exists(paper_file):
        continue
    # all paper content
    paper_content = json.load(open(paper_file))
    # all possible papers
    papers = list(paper_content.keys())
    
    os.makedirs(f'{filename.path}/reviewers', exist_ok=True)
    os.makedirs(f'{filename.path}/papers', exist_ok=True)
    for p in papers:
        with open(f'{filename.path}/papers/{p}.txt', 'w') as f:
            f.write('New paper:\n')
            expertise_paper_info = paper_content.get(p, None)
            if expertise_paper_info is None:
                continue
            f.write(f'<Title: {expertise_paper_info["content"]["title"]}\n')
            f.write(f'Abstract: {expertise_paper_info["content"]["abstract"]} >\n')
    
    for file in os.scandir(reviewer_folder):
        reviewer_published_papers, reviewer_expertise_papers = None, None
        reviewer = int(file.name.strip('.jsonl')[1:])
        with open(file) as f:
            reviewer_published_papers = [json.loads(line) for line in f]
            
            row = df[df['ParticipantID'] == reviewer].iloc[0]
            reviewer_expertise_papers =  {row[f'Paper{x}']: row[f'Expertise{x}'] for x in range(1, 11)
                                            if not pd.isna(row[f'Paper{x}'])}
        
        with open(f'{filename.path}/reviewers/{reviewer}.txt', 'w') as f:
            f.write('Reviewer papers:\n')
            for i, p in enumerate(reviewer_published_papers):
                f.write(f'{i+1}. Title: {p["content"]["title"]}\n')
                f.write(f'Abstract: {p["content"]["abstract"]}\n')
           

### Query language models 

In [ ]:
!pip3 install openai

In [48]:
import os
import openai
from openai import OpenAI
os.environ["OPENAI_API_KEY"] = ""
openai.api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI()
models = client.models.list()
#for model in models.data:
#   print(model.id)


def get_openai_completion(prompt, model="o1-mini"):
   messages = [{"role": "user", "content": prompt}]
   response = client.chat.completions.create(
      model=model,
      messages=messages,
      temperature=1, # this is the degree of randomness of the model's output
   )
   return response.choices[0].message.content

In [ ]:
pip install anthropic

In [23]:
import re
import os
import anthropic

client = anthropic.Anthropic(
  # defaults to os.environ.get("ANTHROPIC_API_KEY")
  api_key="",
)
def get_anthropic_completion(prompt, model='claude-3-5-sonnet-20241022'):
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages
    )
    return response.content[0].text

In [ ]:
cur_dir = os.getcwd()
dic = {}
data_dir = '../evaluation_datasets/'
#prompt_file = 'prompt.txt'
prompt_file_0shot = 'prompt_0shot.txt'
with open(f'{data_dir}/{prompt_file_0shot}', 'r') as f:
    prompt = f.read()
not_exact_outputs = []
for filename in [f'd_20_{i}' for i in range(2, 11)]:
    not_exact_output = 0
    print(filename)
    filename_path = os.path.join(data_dir, filename)
    reviewer_folder = filename_path + '/archives/'
    # all paper content
    score_dic = {}
    for file in os.scandir(reviewer_folder):
        reviewer_dic = {}
        reviewer = int(file.name.strip('.jsonl')[1:])
        with open(f'{filename_path}/reviewers/{reviewer}.txt', 'r') as f:
            reviewer_content = f.read() 
        row = df[df['ParticipantID'] == reviewer].iloc[0]
        reviewer_expertise_papers =  {row[f'Paper{x}']: row[f'Expertise{x}'] for x in range(1, 11)
                                            if not pd.isna(row[f'Paper{x}'])}
        for p in list(reviewer_expertise_papers.keys()):
            with open(f'{filename_path}/papers/{p}.txt', 'r') as f:
                paper_content = f.read()
                full_prompt = f'{prompt}\n<<\nInput: \n{reviewer_content}=====================\n{paper_content}>>'
                with open(f'{data_dir}/test_prompt.txt', 'w') as f:
                    f.write(full_prompt)
            with open(f'{data_dir}/results.txt', 'a') as f:
                results = get_anthropic_completion(full_prompt, model="claude-3-5-sonnet-20241022")
                f.write(f'{reviewer}, {p}\n')
                f.write(results)
                f.write('\n')
                f.write('=====================\n')
                try:
                    score = float(results.split('\n')[0])
                except:
                    score = float(re.sub(r'[^\d]+', '', results.split('\n')[0]))
                    not_exact_output += 1
                reviewer_dic[p] = score
        print('Completed for reviewer:', reviewer)
        score_dic[str(reviewer)] = reviewer_dic
    with open(f'../predictions/claudesonnet35_{filename}_ta.json', 'w') as f:
        json.dump(score_dic, f)
    print('Completed for dataset:', filename)
    not_exact_outputs.append(not_exact_output)
print('Not exact output:', not_exact_outputs)
    


In [ ]:
pip install -q -U google-genai

In [11]:
import os
from google import genai

client = genai.Client(api_key="")
def get_gemini_completion(prompt, model="gemini-2.0-flash"):
    response = client.models.generate_content(
        model=model, contents=prompt)
    return response.text

#r = get_gemini_completion('How is your day?')

In [ ]:
import time
cur_dir = os.getcwd()
dic = {}
data_dir = '../evaluation_datasets/'
#prompt_file = 'prompt.txt'
prompt_file_0shot = 'prompt_0shot.txt'
with open(f'{data_dir}/{prompt_file_0shot}', 'r') as f:
    prompt = f.read()
not_exact_outputs = []
for filename in [f'd_20_{i}' for i in range(1, 11)]:
    not_exact_output = 0
    print(filename)
    filename_path = os.path.join(data_dir, filename)
    reviewer_folder = filename_path + '/archives/'
    # all paper content
    if os.path.exists(f'../predictions/gemini2flash_{filename}_ta.json'):
        score_dic = json.load(open(f'../predictions/gemini2flash_{filename}_ta.json'))
    else:
        score_dic = {}
    for file in os.scandir(reviewer_folder):
        
        reviewer_dic = {}
        reviewer = int(file.name.strip('.jsonl')[1:])
        if str(reviewer) in score_dic:
            print('Already done for reviewer:', reviewer)
            continue
        with open(f'{filename_path}/reviewers/{reviewer}.txt', 'r') as f:
            reviewer_content = f.read() 
        row = df[df['ParticipantID'] == reviewer].iloc[0]
        reviewer_expertise_papers =  {row[f'Paper{x}']: row[f'Expertise{x}'] for x in range(1, 11)
                                            if not pd.isna(row[f'Paper{x}'])}
        for p in list(reviewer_expertise_papers.keys()):
            with open(f'{filename_path}/papers/{p}.txt', 'r') as f:
                paper_content = f.read()
                full_prompt = f'{prompt}\n<<\nInput: \n{reviewer_content}=====================\n{paper_content}>>'
                with open(f'{data_dir}/test_prompt.txt', 'w') as f:
                    f.write(full_prompt)
            with open(f'{data_dir}/results.txt', 'a') as f:
                results = get_gemini_completion(full_prompt, model="gemini-2.0-flash")
                f.write(f'{reviewer}, {p}\n')
                f.write(results)
                f.write('\n')
                f.write('=====================\n')
                try:
                    score = float(results.split('\n')[0])
                except:
                    score = float(re.sub(r'[^\d]+', '', results.split('\n')[0]))
                    not_exact_output += 1
                reviewer_dic[p] = score
        print('Completed for reviewer:', reviewer)
        score_dic[str(reviewer)] = reviewer_dic
        with open(f'../predictions/gemini2flash_{filename}_ta.json', 'w') as f:
            json.dump(score_dic, f)
    with open(f'../predictions/gemini2flash_{filename}_ta.json', 'w') as f:
        json.dump(score_dic, f)
    print('Completed for dataset:', filename)
    not_exact_outputs.append(not_exact_output)
print('Not exact output:', not_exact_outputs)

In [ ]:
pip install -U adapters

In [ ]:
# get cosine similarity between the several papers
from sklearn.metrics.pairwise import cosine_similarity
embeddings = embeddings.detach().numpy()


In [ ]:
for i in range(1, 11):
    filename_path = os.path.join(data_dir, f'd_20_{i}')
    
    with open(f'../predictions/claudesonnet35_d_20_{i}_ta.json', 'r') as f:
        data = json.load(f)
        for r in data.keys():
            for p in data[r].keys():
                try:
                    data[r][p] = float(data[r][p])
                except:
                    with open(f'{filename_path}/reviewers/{r}.txt', 'r') as f:
                        reviewer_content = f.read() 
                    with open(f'{filename_path}/papers/{p}.txt', 'r') as f:
                        paper_content = f.read()
                    full_prompt = f'{prompt}\n<<\nInput: \n{reviewer_content}=====================\n{paper_content}>>'
                    results = get_anthropic_completion(full_prompt, model="claude-3-5-sonnet-20241022")
                    print(results)
                    try:
                        score = float(results.split('\n')[0])
                    except:
                        score = float(re.sub(r'[^\d]+', '', results.split('\n')[0]))
                    data[r][p] = score

                
    with open(f'../predictions/claudesonnet35_d_20_{i}_ta.json', 'w') as f:
        json.dump(data, f)

In [ ]:
participants = set(df['ParticipantID'])

# Getting papers
papers = set()
for x in range(1, 11):
    tmp_papers = set(df[~pd.isna(df[f'Paper{x}'])][f'Paper{x}'])
    papers = papers.union(tmp_papers)
print('# of Papers', len(papers))

# Translating df from csv to dict of the form {participant: {Paper1: Expertise1, Paper2: Expertise2, ...}}
data = {}
for idx, row in df.iterrows():
    key = str(row['ParticipantID'])
    data[key] = {row[f'Paper{x}']: row[f'Expertise{x}'] for x in range(1, 11)
                                                            if not pd.isna(row[f'Paper{x}'])}
    
rev_profiles = {}
for rev in participants:
    with open(f'../data/participants/{rev}.json', 'r') as handler:
        rev_profiles[rev] = json.load(handler)
        rev_profiles[rev]['papers'] = set([rev_profiles[rev]['papers'][i]['paperId'] for i in range(len(rev_profiles[rev]['papers']))])
print('# of Participants', len(rev_profiles))

### SPECTER 2

In [ ]:
from transformers import AutoTokenizer
from adapters import AutoAdapterModel

# load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained('allenai/specter2_base')

#load base model
model = AutoAdapterModel.from_pretrained('allenai/specter2_base')

#load the adapter(s) as per the required task, provide an identifier for the adapter in load_as argument and activate it
model.load_adapter("allenai/specter2", source="hf", load_as="specter2", set_active=True)

papers = [{'title': 'BERT', 'abstract': 'We introduce a new language representation model called BERT'},
          {'title': 'Attention is all you need', 'abstract': ' The dominant sequence transduction models are based on complex recurrent or convolutional neural networks'}]

# concatenate title and abstract
text_batch = [d['title'] + tokenizer.sep_token + (d.get('abstract') or '') for d in papers]
print(text_batch)
# preprocess the input
inputs = tokenizer(text_batch, padding=True, truncation=True,
                                   return_tensors="pt", return_token_type_ids=False, max_length=512)
output = model(**inputs)
# take the first token in the batch as the embedding
embeddings = output.last_hidden_state[:, 0, :]

In [ ]:
import os 
import json
import torch
import pandas as pd
import numpy as np

from transformers import AutoTokenizer
from adapters import AutoAdapterModel
from sklearn.metrics.pairwise import cosine_similarity

# load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained('allenai/specter2_base')

#load base model
model = AutoAdapterModel.from_pretrained('allenai/specter2_base')

#load the adapter(s) as per the required task, provide an identifier for the adapter in load_as argument and activate it
model.load_adapter("allenai/specter2", source="hf", load_as="specter2", set_active=True)

cur_dir = os.getcwd()
dic = {}
data_dir = '../evaluation_datasets/'

df = pd.read_csv('../data/evaluations.csv', sep='\t')
# get the papers
for filename in [f'd_20_{i}' for i in range(1, 11)]:
    print(filename)
    filename_path = os.path.join(data_dir, filename)
    reviewer_folder = filename_path + '/archives/'
    paper_file = filename_path + '/submissions.json'
    
    if os.path.exists(f'../predictions/spector2_all_{filename}_ta.json'):
        score_dic = json.load(open(f'../predictions/spector2_all_{filename}_ta.json'))
    else:
        score_dic = {}
    if not os.path.exists(paper_file):
        continue
    # all paper content
    paper_content = json.load(open(paper_file))
    
    # get paper embeddings
    papers = list(paper_content.keys())
    paper_ta = {p: {'title': paper_content[p]['content']['title'], 'abstract':paper_content[p]['content']['abstract']} for p in papers}
    
    for file in os.scandir(reviewer_folder):
        
        reviewer = int(file.name.strip('.jsonl')[1:])
        if str(reviewer) in score_dic:
            print('Already done for reviewer:', reviewer)
            continue
        with open(file) as f:
            reviewer_published_papers = [json.loads(line) for line in f]
            published_paper_content = {p['id']: {'title': p['content']['title'], 'abstract': p['content']['abstract']} for p in reviewer_published_papers}
            
        row = df[df['ParticipantID'] == reviewer].iloc[0]
        reviewer_expertise_papers =  [row[f'Paper{x}'] for x in range(1, 11)if not pd.isna(row[f'Paper{x}'])]
        expertise_paper_content = {p: {'title': paper_content[p]['content']['title'], 'abstract':paper_content[p]['content']['abstract']} for p in reviewer_expertise_papers}

        p_batch = [d['title'] + tokenizer.sep_token + (d.get('abstract') or '') for d in published_paper_content.values()]
        e_batch = [d['title'] + tokenizer.sep_token + (d.get('abstract') or '') for d in expertise_paper_content.values()]

        expertise_input = tokenizer(e_batch, padding=True, truncation=True,
                                   return_tensors="pt", return_token_type_ids=False, max_length=512)
        e_output = model(**expertise_input)
        e_embeddings = e_output.last_hidden_state[:, 0, :]

        published_input = tokenizer(p_batch, padding=True, truncation=True,
                                   return_tensors="pt", return_token_type_ids=False, max_length=512)
        p_output = model(**published_input)
        p_embeddings = p_output.last_hidden_state[:, 0, :]

        # get cosine simliarity
        sim = cosine_similarity(e_embeddings.detach().numpy(), p_embeddings.detach().numpy())
        reviewer_dic = {str(p): [float(j) for j in s] for p, s in zip(expertise_paper_content.keys(), sim)}
        score_dic[str(reviewer)] = reviewer_dic

        print('Completed for reviewer:', reviewer)
        score_dic[str(reviewer)] = reviewer_dic
        with open(f'../predictions/spector2_all_{filename}_ta.json', 'w') as f:
            json.dump(score_dic, f)
    with open(f'../predictions/spector2_all_{filename}_ta.json', 'w') as f:
        json.dump(score_dic, f)
    print('Completed for dataset:', filename)

In [ ]:
score_dic

